# Lab 4. Bayesian network: student performance

Build a Bayesian network by hand, estimate its probabilities from data and use it to answer questions about how lifestyle, family and motivation affect school grades.

- **Data:** [Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance), UCI (Cortez & Silva, 2008), Portuguese language course, 649 students of two Portuguese secondary schools.
- **Tool:** `pgmpy` — a discrete Bayesian network, parameters estimated from data, inference by variable elimination.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator
from pgmpy.structure_score import BIC

pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 200)
plt.rcParams["figure.dpi"] = 72

## 1. Data

**One object** is one student: a questionnaire about family and lifestyle plus the grades for the year. Out of 33 columns, 8 are used — those between which dependencies can be argued without special knowledge:

| Variable | Column | Meaning |
| --- | --- | --- |
| `sex` | `sex` | F / M |
| `going_out` | `goout` | going out with friends, 1 (very low) – 5 (very high) |
| `weekend_alcohol` | `Walc` | weekend alcohol consumption, 1 – 5 |
| `mother_edu` | `Medu` | mother's education: 0 none … 4 higher |
| `past_failures` | `failures` | number of previously failed classes |
| `wants_higher` | `higher` | wants to take higher education (yes / no) |
| `study_time` | `studytime` | weekly study time: 1 (< 2 h), 2 (2–5 h), 3 (5–10 h), 4 (> 10 h) |
| `final_grade` | `G3` | final grade, 0–20 |

**Expected dependencies:**

- boys and sociable students drink more;
- drinking, sex and motivation affect study time;
- educated mothers raise ambitions and help to avoid failures;
- past failures, study time and motivation determine the grade.

In [ ]:
raw = pd.read_csv("data/student-por.csv", sep=";")
COLUMNS = ["sex", "goout", "Walc", "Medu", "failures", "higher", "studytime", "G3"]
print(f"students: {len(raw)}, columns: {raw.shape[1]}, missing: {raw.isna().sum().sum()}")
print(f"duplicates: {raw.duplicated().sum()}")
pd.Series({c: raw[c].value_counts().sort_index().to_dict() for c in COLUMNS}).to_frame("counts")

## 2. Preparation

- **No missing values and no duplicates.** Nothing to impute.
- **Few states per variable.** A conditional probability table needs data for every combination of parent states. With 649 students a variable with three 3-state parents already has 27 combinations, so every variable is cut into 2–3 states.
- **Rare categories are merged:** `Medu = 0` (6 students) goes into "basic"; 2 and 3 failures (16 and 14 students) become "yes"; `studytime = 4` (35) is joined with 3; `Walc = 5` (45) is joined with 4.

| Variable | States | Why |
| --- | --- | --- |
| `going_out` | low (1–2), medium (3), high (4–5) | three groups of similar size (193 / 205 / 251) |
| `weekend_alcohol` | low (1), moderate (2–3), high (4–5) | "never drinks" vs "sometimes" vs "a lot" |
| `mother_edu` | basic (≤ 9th grade), secondary, higher | the three school levels |
| `study_time` | < 2 h, 2–5 h, > 5 h | the original codes, with the rare top level merged |
| `final_grade` | **fail** (< 10), **pass** (10–13), **good** (≥ 14) | 10 is the pass mark of the Portuguese 0–20 scale; 14+ is the usual "good" level and covers the top 30% |

The 15 students with `G3 = 0` did not pass the course (most likely did not take the final exam), so they are counted as **fail**: for the questions below "did not get a passing grade" is exactly what matters.

In [ ]:
STATES = {
    "sex": ["F", "M"],
    "going_out": ["low", "medium", "high"],
    "weekend_alcohol": ["low", "moderate", "high"],
    "mother_edu": ["basic", "secondary", "higher"],
    "past_failures": ["no", "yes"],
    "wants_higher": ["no", "yes"],
    "study_time": ["<2h", "2-5h", ">5h"],
    "final_grade": ["fail", "pass", "good"],
}
students = pd.DataFrame(
    {
        "sex": raw["sex"],
        "going_out": pd.cut(raw["goout"], [0, 2, 3, 5], labels=STATES["going_out"]),
        "weekend_alcohol": pd.cut(raw["Walc"], [0, 1, 3, 5], labels=STATES["weekend_alcohol"]),
        "mother_edu": pd.cut(raw["Medu"], [-1, 2, 3, 4], labels=STATES["mother_edu"]),
        "past_failures": raw["failures"].gt(0).map({False: "no", True: "yes"}),
        "wants_higher": raw["higher"],
        "study_time": pd.cut(raw["studytime"], [0, 1, 2, 4], labels=STATES["study_time"]),
        "final_grade": pd.cut(raw["G3"], [-1, 9, 13, 20], labels=STATES["final_grade"]),
    }
).astype(str)

pd.concat(
    {c: students[c].value_counts().reindex(s) for c, s in STATES.items()},
    names=["variable", "state"],
).to_frame("students")

## 3. Questions

Formulated before building the network:

1. **Q1.** How much does a past failure raise the risk of failing the final grade? $P(\text{fail})$ vs $P(\text{fail} \mid \text{past failures})$.
2. **Q2.** A boy studies less than 2 hours a week. How likely is it that he drinks heavily at weekends? $P(\text{alcohol} = \text{high})$ vs $P(\text{alcohol} = \text{high} \mid \text{male}, \text{study} < 2\text{h})$ — the evidence is an *effect* of drinking, so the network has to reason backwards.
3. **Q3.** What does a failed grade tell us about the student? Compare $P(\text{no higher-education plans})$ and $P(\text{past failures})$ before and after seeing the failed grade.

## 4. Network structure

| Edge | Why there is a dependency | Why this direction |
| --- | --- | --- |
| `sex` → `weekend_alcohol` | boys drink more often | sex is fixed from birth |
| `going_out` → `weekend_alcohol` | drinking happens at parties | going out is the occasion for drinking |
| `sex` → `study_time` | girls usually spend more time on homework | sex comes first |
| `weekend_alcohol` → `study_time` | party weekends take time and energy from studying | the weekend comes before the next study week |
| `wants_higher` → `study_time` | students aiming at university work more | the goal motivates the effort |
| `mother_edu` → `wants_higher` | educated parents pass on the value of education | the mother's education was completed long before |
| `mother_edu` → `past_failures` | educated parents can help with homework | same: it precedes the student's school history |
| `past_failures` → `wants_higher` | failing a class can discourage further study | the failures are in the past |
| `past_failures` → `final_grade` | gaps in knowledge stay | the past precedes the current grade |
| `study_time` → `final_grade` | more practice, better result | studying happens before the exam |
| `wants_higher` → `final_grade` | motivation helps beyond hours: attention in class, exam preparation | the goal exists before the grade |

**Deliberately not connected:** alcohol and the mother's education have no direct edge to the grade. They are assumed to act through study time, ambitions and past failures. Section 5 checks this assumption against the data.

In [ ]:
EDGES = [
    ("sex", "weekend_alcohol"),
    ("going_out", "weekend_alcohol"),
    ("sex", "study_time"),
    ("weekend_alcohol", "study_time"),
    ("wants_higher", "study_time"),
    ("mother_edu", "wants_higher"),
    ("mother_edu", "past_failures"),
    ("past_failures", "wants_higher"),
    ("past_failures", "final_grade"),
    ("study_time", "final_grade"),
    ("wants_higher", "final_grade"),
]
POSITIONS = {
    "going_out": (1.5, 3),
    "mother_edu": (4, 3),
    "past_failures": (6.5, 3),
    "sex": (-0.5, 2),
    "weekend_alcohol": (1.5, 2),
    "wants_higher": (5, 2),
    "study_time": (2.5, 1),
    "final_grade": (5, 0),
}


def fit(edges: list[tuple[str, str]], ess: float = 5) -> DiscreteBayesianNetwork:
    model = DiscreteBayesianNetwork(edges)
    estimator = DiscreteBayesianEstimator(state_names=STATES, equivalent_sample_size=ess)
    return model.fit(students, estimator=estimator)


def draw(model: DiscreteBayesianNetwork) -> None:
    _, ax = plt.subplots(figsize=(10, 5.5))
    nx.draw_networkx_nodes(
        model, POSITIONS, ax=ax, node_size=4200, node_color="#d1e5f0", edgecolors="#2166ac"
    )
    nx.draw_networkx_edges(
        model, POSITIONS, ax=ax, node_size=4200, arrowsize=18, edge_color="#555555", width=1.5
    )
    nx.draw_networkx_labels(
        model, POSITIONS, labels={n: n.replace("_", "\n") for n in model}, ax=ax, font_size=10
    )
    ax.margins(0.12)
    ax.axis("off")
    plt.show()


model = fit(EDGES)
draw(model)
print(f"DAG: {nx.is_directed_acyclic_graph(model)}, nodes: {len(model)}, edges: {len(EDGES)}")

pd.DataFrame(
    {
        node: {
            "parents": ", ".join(model.get_parents(node)),
            "parent combinations": students.groupby(model.get_parents(node)).ngroups,
            "min students per combination": students.groupby(model.get_parents(node)).size().min(),
        }
        for node in model.nodes
        if model.get_parents(node)
    }
).T

**Is the data enough?** `weekend_alcohol`, `wants_higher` and `past_failures` have 3–6 parent combinations, and each has at least 7 students. `study_time` and `final_grade` have 12 combinations, and some are rare: the smallest has only 2 students.

Plain frequencies would be unreliable there. The parameters are therefore estimated with a **Bayesian estimator** (`BDeu` prior, equivalent sample size 5). It adds 5 "virtual" students spread evenly over all cells of a table, so rare combinations get probabilities pulled towards uniform instead of 0 or 1. Frequent combinations are practically unaffected.

## 5. Conditional probability tables

In [ ]:
def cpt(model: DiscreteBayesianNetwork, node: str) -> pd.DataFrame:
    cpd = model.get_cpds(node)
    parents = cpd.variables[1:]
    columns = pd.MultiIndex.from_product([cpd.state_names[p] for p in parents], names=parents)
    table = pd.DataFrame(cpd.get_values(), index=cpd.state_names[node], columns=columns).T
    return table.assign(students=students.groupby(parents).size().reindex(table.index).fillna(0))


cpt(model, "weekend_alcohol")

In [ ]:
cpt(model, "wants_higher")

In [ ]:
cpt(model, "study_time")

In [ ]:
cpt(model, "final_grade")

**Findings** (the `students` column is the number of real students behind each row)

- **Weekend alcohol:** the share of heavy drinkers ranges from **4% (girls who rarely go out) to 64% (boys who go out a lot)**. The two parents **interact**: going out a lot raises heavy drinking by 15 points for girls but by 50 points for boys. Such an interaction is exactly what a CPT can store and a simple additive model cannot.
- **Ambitions:** without past failures 89–99% want higher education, more with a better-educated mother. A past failure lowers it, most strongly in families with basic education (62%). The 95% for "higher education + failure" rests on only 7 students.
- **Study time:**
  - motivated girls who do not drink study the most (30% study more than 5 h);
  - motivated heavy-drinking boys study the least of the motivated groups (52% less than 2 h);
  - among motivated students (the large groups) more drinking and being a boy both shift study time down, as expected.
  
  Among unmotivated students the pattern is irregular (e.g. non-drinking boys study least), but these rows have only 2–18 students and are dominated by noise and the prior, so they should not be interpreted.
- **Final grade:**
  - For motivated students without failures, more study time steadily lowers the risk of failing (11% → 8% → 2%) and raises "good" grades (24% → 39% → 49%).
  - **With past failures, more study time does not help** (fail 41% → 47% → 49%). This is unexpected. A likely explanation is reverse causality: struggling students are made to study more (by parents, tutors), so long study time is partly a symptom of difficulties. Also, only 8 students are in the last group.

### Does the model agree with the data?

In [ ]:
inference = VariableElimination(model)


def query(
    node: str, evidence: dict[str, str], engine: VariableElimination = inference
) -> pd.Series:
    factor = engine.query([node], evidence=evidence or None, show_progress=False)
    return pd.Series(factor.values, index=factor.state_names[node]).reindex(STATES[node])


def observed(node: str, evidence: dict[str, str]) -> tuple[pd.Series, int]:
    subset = students.loc[(students[list(evidence)] == pd.Series(evidence)).all(axis=1), node]
    return subset.value_counts(normalize=True).reindex(STATES[node]).fillna(0), len(subset)


CHECKS = [
    ("final_grade", "fail", {"past_failures": "yes"}),
    ("final_grade", "fail", {"wants_higher": "no"}),
    ("final_grade", "fail", {"study_time": "<2h"}),
    ("final_grade", "fail", {"sex": "M"}),
    ("final_grade", "fail", {"going_out": "high"}),
    ("final_grade", "fail", {"weekend_alcohol": "high"}),
    ("final_grade", "good", {"weekend_alcohol": "high"}),
    ("final_grade", "good", {"mother_edu": "higher"}),
    ("past_failures", "yes", {"mother_edu": "basic"}),
]
rows = []
for node, state, evidence in CHECKS:
    data, n = observed(node, evidence)
    ((key, value),) = evidence.items()
    rows.append(
        {
            "P(A | E)": f"P({node}={state} | {key}={value})",
            "model": query(node, evidence)[state],
            "data": data[state],
            "students": n,
        }
    )
check = pd.DataFrame(rows).set_index("P(A | E)")
check["difference"] = check["model"] - check["data"]
check.round(3)

In [ ]:
without_edge = VariableElimination(
    fit([edge for edge in EDGES if edge != ("mother_edu", "past_failures")])
)
display(
    pd.DataFrame(
        {
            education: {
                "data": observed("past_failures", {"mother_edu": education})[0]["yes"],
                "graph without mother_edu -> past_failures": query(
                    "past_failures", {"mother_edu": education}, without_edge
                )["yes"],
                "chosen graph": query("past_failures", {"mother_edu": education})["yes"],
            }
            for education in STATES["mother_edu"]
        }
    ).rename_axis("P(past_failures = yes | mother_edu)")
)

bic = BIC(students)
pd.Series(
    {
        "chosen graph": bic.score(model),
        "+ weekend_alcohol -> final_grade": bic.score(
            DiscreteBayesianNetwork([*EDGES, ("weekend_alcohol", "final_grade")])
        ),
        "+ mother_edu -> final_grade": bic.score(
            DiscreteBayesianNetwork([*EDGES, ("mother_edu", "final_grade")])
        ),
    },
    name="BIC (higher is better)",
).to_frame()

**Findings: model vs data**

- **Edges that exist in the graph are reproduced well.** For the direct parents of the grade (past failures, ambitions, study time) the model differs from the raw frequencies by at most 0.012. The same holds for `mother_edu → past_failures`. The first table shows why the edge `mother_edu → past_failures` is there. Without it the graph has to give everyone the same failure rate (0.157), which **overestimates** failures for children of higher-educated mothers (data: 0.040). The edge is justified by the domain (parents help with homework, section 4), and the data confirms it.
- **Two dependencies are weaker in the model than in the data:**
  - *Alcohol and the grade:* the model says 26% "good" among heavy drinkers, the data says 17%. Heavy drinking is associated with worse grades even beyond what shorter study time explains.
  - *Mother's education and the grade:* the model says 35% "good" for students of higher-educated mothers, the data says 46%. Their children do better than the paths through ambitions and failures explain.
- **Why these edges are not added anyway.** `final_grade` already has 3 parents and 12 combinations. A fourth 3-state parent makes 36 combinations for 649 students, many of them nearly empty. BIC weighs the fit against the number of parameters, and it drops from −4263 to about −4397 for either edge. The extra parameters cost more than the better fit gives. With this amount of data the smaller graph is the better model, and the two weak links are a known limitation.

In [ ]:
pd.DataFrame(
    {
        f"ess = {ess}": query(
            "final_grade",
            {"past_failures": "yes", "wants_higher": "no"},
            VariableElimination(fit(EDGES, ess)),
        )
        for ess in [1, 5, 20]
    }
).T

**Sensitivity to the prior.** For a rare combination (past failure and no ambitions, 33 students) the large probabilities barely depend on the equivalent sample size (fail 0.54–0.57), but the tiny probability of "good" moves from 0.003 to 0.048. The prior matters only for small probabilities in sparse cells. The conclusions below rely on the large probabilities, so they do not depend on the choice ess = 5.

## 6. Inference

In [ ]:
QUESTIONS = [
    ("Q1", "final_grade", "fail", {"past_failures": "yes"}),
    ("Q2", "weekend_alcohol", "high", {"sex": "M"}),
    ("Q2", "weekend_alcohol", "high", {"sex": "M", "study_time": "<2h"}),
    ("Q3", "wants_higher", "no", {"final_grade": "fail"}),
    ("Q3", "past_failures", "yes", {"final_grade": "fail"}),
]
answers = pd.DataFrame(
    [
        {
            "question": q,
            "event": f"{node} = {state}",
            "evidence": ", ".join(f"{k} = {v}" for k, v in evidence.items()),
            "prior P(A)": query(node, {})[state],
            "posterior P(A | E)": query(node, evidence)[state],
            "data": observed(node, evidence)[0][state],
        }
        for q, node, state, evidence in QUESTIONS
    ]
).set_index("question")
answers["lift"] = answers["posterior P(A | E)"] / answers["prior P(A)"]
answers

**Findings**

- **Q1.** A past failure raises the risk of failing from **16% to 49%**, three times higher (lift 3.1). This is the strongest single factor. In the model-vs-data table the next ones are no ambitions (47%) and short study time (22%). In the model, sex, going out and alcohol barely change it, but in the data heavy drinking raises it to 21% — the link the model underestimates (section 5).
- **Q2.** Heavy weekend drinking: 20% overall, **35% for boys** (sex is a direct parent), and **40%** for a boy who studies less than 2 hours (data: 36% and 42%). The second step is reasoning backwards: the network inverts the arrow `weekend_alcohol → study_time` with Bayes' rule, so short study time becomes evidence for one of its causes. Most of the increase comes from sex; study time adds 5 more points.
- **Q3 (backwards).** After seeing a failed grade:
  - "no higher-education plans" rises from 11% to **33%**;
  - "failed before" rises from 16% to **49%**.
  
  The **lifts are almost equal (×3.0 and ×3.1)**: a failed grade is equally strong evidence for both. "Failed before" ends up higher only because it was more common to begin with. This is a statement about probabilities, not about the *reason* of the failure.
- All posteriors agree with the raw frequencies within about 0.02.

## 7. Experiment with evidence

In [ ]:
steps = [
    ("nothing known", {}),
    ("+ going_out = high", {"going_out": "high"}),
    ("+ sex = M", {"going_out": "high", "sex": "M"}),
    ("+ wants_higher = no", {"going_out": "high", "sex": "M", "wants_higher": "no"}),
    (
        "+ past_failures = yes",
        {"going_out": "high", "sex": "M", "wants_higher": "no", "past_failures": "yes"},
    ),
]
experiment = pd.DataFrame({name: query("final_grade", evidence) for name, evidence in steps}).T

fig, ax = plt.subplots(figsize=(9, 3.5))
experiment.iloc[::-1].plot.barh(
    stacked=True, color=["#d6604d", "#f7f7f7", "#4393c3"], edgecolor="#555555", ax=ax
)
ax.set_xlabel("P(final_grade)")
ax.legend(title="final_grade", bbox_to_anchor=(1, 1))
plt.show()
experiment

In [ ]:
short_study = {"study_time": "<2h"}
short_study_drinking = {"study_time": "<2h", "weekend_alcohol": "high"}
print(f"P(wants_higher=no | <2h)       = {query('wants_higher', short_study)['no']:.3f}")
print(f"P(wants_higher=no | <2h, high) = {query('wants_higher', short_study_drinking)['no']:.3f}")
pd.DataFrame(
    {
        "model: study_time = <2h": query("final_grade", short_study),
        "model: + weekend_alcohol = high": query("final_grade", short_study_drinking),
        "data: study_time = <2h": observed("final_grade", short_study)[0],
        "data: + weekend_alcohol = high": observed("final_grade", short_study_drinking)[0],
    }
).T

**Findings: adding evidence step by step** (target: `final_grade`)

1. **Nothing known:** fail 16%, pass 55%, good 30%.
2. **going_out = high:** almost no change (fail 15.7% → 15.8%). In the graph, going out reaches the grade only through the long chain `going_out → weekend_alcohol → study_time → final_grade`, and every link dilutes the effect. In the data the effect is somewhat larger (19.5%, section 5) — part of it goes through paths the graph does not have.
3. **+ sex = M:** slightly worse (good 29% → 27%). Sex acts through the same chain and directly on study time.
4. **+ wants_higher = no:** a jump to fail **48%**, good 3%. Ambition is a direct parent of the grade and also lowers study time.
5. **+ past_failures = yes:** fail **58%**, good 1%. Another direct parent.

Evidence about the parents of a variable changes it strongly, evidence that has to travel along long chains changes it weakly.

**Explaining away.** Among students who study less than 2 hours, also learning that the student drinks a lot makes failing slightly *less* likely: 22% → 19% (data: 24% → 20%).

- Short study time has several possible causes. If heavy drinking already explains it, the other cause, "no ambitions", becomes less likely (21% → 13%, printed above the table).
- Ambitions directly affect the grade, so the predicted grade improves.

This follows from the collider `weekend_alcohol → study_time ← wants_higher`. In the data the effect holds for "fail" but not for "good" (18% → 13%), because heavy drinking also hurts the grade directly — the link the model is missing.

## 8. Structure analysis

In [ ]:
reduced = fit([edge for edge in EDGES if edge != ("past_failures", "final_grade")])
reduced_inference = VariableElimination(reduced)
pd.DataFrame(
    {
        "full graph": query("final_grade", {"past_failures": "yes"}),
        "without past_failures -> final_grade": query(
            "final_grade", {"past_failures": "yes"}, reduced_inference
        ),
        "data": observed("final_grade", {"past_failures": "yes"})[0],
    }
).T

**Findings: removing the edge `past_failures → final_grade`**

- **The answer changed a lot.** With the full graph $P(\text{fail} \mid \text{past failures}) = 0.49$, equal to the raw frequency. Without the edge it drops to **0.24**, barely above the prior 0.16. Only the path through ambitions is left, and it carries a small part of the effect.
- **Why the structure matters:** the network can express only the dependencies its edges allow. Without the edge, the model assumes that, given ambitions and study time, the grade does not depend on past failures. The CPTs are estimated under that assumption, and inference can only use the paths that remain.
- **Can directions be chosen from data alone? Only partly.**
  - A chain `A → B → C`, the reversed chain and a fork `A ← B → C` imply the same independences and fit any data equally well.
  - Only colliders `A → B ← C` leave a distinct trace: A and C become dependent once B is known.
  
  The directions here therefore come from time order and common sense, not from the data.

## 9. Is this a causal model?

**Not proven.** The network is a compact description of the joint distribution, and an arrow only says how the probabilities were factorized. The same data fits other graphs with reversed arrows. Section 5 even shows a possible reverse effect: weak students study more *because* they are weak.

Some arrows can be called **plausibly causal**, based on time order and knowledge of the domain, not on the data:

- `sex → weekend_alcohol` and `sex → study_time`: sex is fixed before any behavior, so the reverse is impossible. Even so, the "cause" is social norms rather than biology.
- `past_failures → final_grade` and `mother_edu → past_failures`: the past cannot be caused by the future. But a hidden common cause (ability, family income) can produce both, so part of the dependency may be confounding.

Edges such as `weekend_alcohol → study_time` or `wants_higher → study_time` can be confounded by unmeasured traits such as personality or parental control. A causal claim would require an intervention, e.g. randomly assigning extra tutoring and comparing grades.

## 10. Conclusions

- **What we learned:**
  - past failures triple the risk of failing (16% → 49%);
  - lack of ambition is almost as strong (47%);
  - heavy drinking is associated with lower grades more strongly than the graph allows;
  - heavy weekend drinking ranges from 4% to 64% depending on sex and going out, with a strong interaction between the two.
- **Most interesting dependencies:**
  - extra study time does not help students with past failures (probably reverse causality);
  - explaining away between alcohol and ambitions through study time;
  - the equal ×3 lift of both explanations of a failed grade.
- **Evidence:** facts about the direct parents of a variable change it strongly, facts that reach it through long chains change it weakly. Observing a common effect creates a dependency between its causes.
- **Dependence on structure:** removing one edge changed the answer to Q1 from 0.49 to 0.24, and adding `mother_edu → past_failures` fixed a 0.12 mismatch with the data. Inference is only as good as the graph.
- **Limitations:**
  - 649 students and sparse parent combinations (smoothed by the prior);
  - coarse discretization;
  - directions chosen by hand;
  - the direct effects of alcohol and of the mother's education on the grade are underestimated (adding them costs too many parameters for this sample);
  - hidden confounders mean the network describes associations, not proven causes.